# Read Waveform Data
This is kind of variant 3.  It uses a set of functions in a new module with the name "s3prototypes".  For this test that should be located in the current directory. 

First, is a variant of an earlier notebook to use the version of "fetch_s3_client" from the new module. Note for now I copied a new module in MsPASS called db_utils.py to this directory and modified s3prototypes to use that local copy.   The reason that was needed is that the version of mspass running on geolab is stale.   When Earthscope folks fix that problem I will need to revert s3prototypes to the original now commented out

In [1]:
from mspasspy.client import Client
mspass_client=Client()
db=mspass_client.get_database("ES2026")
dask_client = mspass_client.get_scheduler()

2026-06-16 11:36:05,526 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='geolab.earthscope.cloud', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/tornado/websocket.py", line 938, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/opt/conda/lib/python3.10/site-packages/tornado/web.py", line 3301, in wrapper
    return method(self, *args, **kwargs)
  File "/opt/conda/lib/python3.10/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the app with a larger value for --session-token-expiration if necessary
2026-06-16 11:36:15,147 - tornado.application - ERROR -

In [2]:
dask_client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/auth0%7C66620230da0803120dd94aaf/proxy/8787/status,
Dashboard: /user/auth0%7C66620230da0803120dd94aaf/proxy/8787/status,Workers: 4
Total threads: 4,Total memory: 29.08 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37355,Workers: 0
Dashboard: /user/auth0%7C66620230da0803120dd94aaf/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:41389,Total threads: 1
Dashboard: /user/auth0%7C66620230da0803120dd94aaf/proxy/34203/status,Memory: 7.27 GiB
Nanny: tcp://127.0.0.1:35025,


In [3]:
from s3_worker_plugin import AsyncS3Worker

BUCKET = "earthscope-mseed-res-na3mtd4fq5kz7pntcyr1uh46use2a--ol-s3"
s3plugin = AsyncS3Worker()
#dask_client.register_plugin(s3plugin)

In [4]:
dask_client.register_plugin(s3plugin)

{'tcp://127.0.0.1:33327': {'status': 'OK'},
 'tcp://127.0.0.1:35045': {'status': 'OK'},
 'tcp://127.0.0.1:41389': {'status': 'OK'},
 'tcp://127.0.0.1:43501': {'status': 'OK'}}

Fetch only data suitable for teleseismic P wave processing for receiver functions.  Use the "delta" attribute from assoc to limit data to epicentral distances between 30 and 95.   Note in this notebook that does nothing as we already filtered input that way, but shows an alternative way to do tha with MongoDB.

In [5]:
query={"delta" : {"$gte" : 30.0, "$lte" : 95.0}}
cursor=db.arrival_css30.find(query)
doclist = list(cursor)
print("Size of arrival table loaded = ",len(doclist))  # validate this is reasonable

Size of arrival table loaded =  82848


In [6]:
# reader we use requires a DataFrame as input 
import pandas as pd
df = pd.DataFrame(doclist)
print(df)

                            _id      evid      orid      arid phase   delta  \
0      6a2a9656f7987912cffda726  298930.0  472052.0  24259456     P  70.912   
1      6a2a9656f7987912cffda727  298930.0  472052.0  24259457     P  71.152   
2      6a2a9656f7987912cffda728  298930.0  472052.0  24259458     P  71.325   
3      6a2a9656f7987912cffda729  298930.0  472052.0  24259459     P  71.991   
4      6a2a9656f7987912cffda72a  298930.0  472052.0  24259460     P  72.167   
...                         ...       ...       ...       ...   ...     ...   
82843  6a2a9657f7987912cffeed83  299892.0  473209.0  24384852     P  60.363   
82844  6a2a9657f7987912cffeed84  299892.0  473209.0  24384853     P  60.761   
82845  6a2a9657f7987912cffeed85  299892.0  473209.0  24384854     P  60.834   
82846  6a2a9657f7987912cffeed86  299892.0  473209.0  24384855     P  61.497   
82847  6a2a9657f7987912cffeed87  299892.0  473209.0  24384856     P  65.421   

         seaz    esaz  timeres iphase  fm  source_l

To keep this dataset within the bounds of GeoLab quotas for this class we need to limit this further.   There are a lot of picks in the ANF data for small events that challenge RF deconvolution algorithms anyway, so we will impose a magnitude filter.   A problem is that the "arrival_css30" collection does not contain a magnitude number but only a link to the source collection.  We will use the pandas api to address this nasty problem.  This is a specialized solution for this exercise, but is hopefully informative to students.

In [7]:
# first load up a dictionary of magnitudes keyed by orid
cursor=db.source.find({})
mags = dict()
for doc in cursor:
    orid=int(doc["orid"])   # int may not be necessary but safer
    if "magnitude" in doc:
        mags[orid] = doc["magnitude"]
    else:
        mags[orid] = -99.

In [8]:
# create a numpy array to hold mag values for each df row and load it with mags values
import numpy as np
N=len(df)
mag_column = np.zeros(N)
i=0
for row in df.itertuples():
    orid = int(row.orid)
    if orid in mags:
        mag_column[i] = mags[orid]
    else:
        mag_column[i] = -99.
    i += 1
df["magnitude"] = mag_column

The reader function we will use below requires the DataFrame key "time" to define the reference time it is to use to define waveform segments.   For clarity I chose to set the arrival time from the ANF pick table to "Ptime".   Hence, we need this trivial patch from the pandas API.

In [9]:
df = df.rename(columns={"Ptime" : "time"})

In [10]:
# now limit by magnitude 
minimum_magnitude = 5.5
df = df.query(f"magnitude >= {minimum_magnitude}")
print("Size of subsetted DataFrame=",len(df))

Size of subsetted DataFrame= 17336


## Run reader
The ugly work for all this inside a function define in what is currently a local module called *s3_segment_reader*.   

blah blah - need to get this done first

In [11]:
import importlib
import s3segment_reader as s3sr
importlib.reload(s3sr)
process_docs = s3sr.run_by_days(df,
                               -240,
                               300,
                               db,
                               auxkeys=["evid","orid","arid","iphase","delta","pick_channel"],
                               )

In [12]:
len(process_docs)

9493

## Test retrieval function
Now run the function that reformats the arrival list to a document list that can be used to drive a parallel retrieve operation with dask.

In [13]:
from mspasspy.util.seismic import print_metadata
# verify doc structure is as expected
print_metadata(process_docs[0])

{
  "net": "AZ",
  "sta": "MONP2",
  "channel_select": "B*",
  "s3objects": [
    "miniseed/AZ/2010/060/MONP2.AZ.2010.060"
  ],
  "arrival": [
    {
      "time": 1267412206.06108,
      "evid": 298936.0,
      "orid": 472058.0,
      "arid": 24260846,
      "iphase": "P",
      "delta": 79.438,
      "pick_channel": "BHZ"
    },
    {
      "time": 1267430467.60053,
      "evid": 298948.0,
      "orid": 472072.0,
      "arid": 24263873,
      "iphase": "P",
      "delta": 78.864,
      "pick_channel": "BHZ"
    }
  ],
  "start": [
    1267411866.06108,
    1267430127.60053
  ],
  "end": [
    1267412606.06108,
    1267430867.60053
  ]
}


In [14]:
from mspasspy.util.seismic import number_live
from mspasspy.algorithms.bundle import bundle_seed_data
from obspy import UTCDateTime
def save_segment_outputs(ens,db,outdir="wf_TimeSeries"):
    """
    Completion function to save outputs to database using a single, common file to store all. 
    Default creates this as a raw binary file of a gazillion doubles.   

    Uses a single Database instance (db) as this function will run under this notebook's 
    interpreter.   This could be a bottleneck, but a volatile test below will verify that. 

    Returns a tuple with (ensemble_size, number_live, elog ) as content where elog 
    is the ErrorLogger attribute of ens.   In this context it is rarely empty but 
    content can be used to appraise failures.  
    """
    nlive = number_live(ens)
    nsize = len(ens.member)
    if nlive>0:
        t = UTCDateTime(ens.member[0].t0)
        dfile = f"{t.year}_{t.julday}.dat"
    else:
        # this should never appear as an actual file
        dfile = "BADDATA_DO_NOT_SAVE"
    ens = db.save_data(ens,
                        collection="wf_TimeSeries",
                        storage_mode="file",
                        dir=outdir,
                        dfile=dfile,
                        return_data=True,    # Default throws ens away
                    )
    return (nsize,nlive,ens.elog)
    

In [15]:
from mspasspy.workflow import sliding_window_pipeline

# Serial test section
The next few cells will be deleted before release.   I used this to debug the s3_segmenter code and related functions that were a bear to fix after Earthscope changed to use of an async client.   The cells are disabled as raw which works with jupyter but causes problems if this is converted to a pythons script.  

In [16]:
synchronous_s3_segmenter = s3sr.make_sync(s3sr.s3_segmenter)

## Parallel section
This should now work - add text when it does

In [17]:
import time
t0=time.time()
print("Submitting retrieve to dask cluster")
out = sliding_window_pipeline(process_docs,
                              synchronous_s3_segmenter,
                              dask_client,
                              completion_function=save_segment_outputs,
                              cfunc_args=[db],
                             #verbose=True,
                             #progress_report_interval=1000,
                             )
t=time.time()
print("Elapsed time=",t-t0)
print("Number of items handled=",len(process_docs))

Submitting retrieve to dask cluster
Elapsed time= 98.83786034584045
Number of items handled= 9493


In [18]:
# get grand totals 
nhandled=0
nlive=0
n_with_errors=0
for nh,nl,elog in out:
    nhandled += nh
    nlive += nl
    if elog.size() > 0:
        n_with_errors += 1
print("Retrieved a total of ",nhandled," waveform segments")
print("Number TimeSeries objects saved as live data in the working database=",nlive)
print("Number of results with errors logged=",n_with_errors)

Retrieved a total of  21  waveform segments
Number TimeSeries objects saved as live data in the working database= 21
Number of results with errors logged= 0


In [19]:
len(out)

9493

In [20]:
print("List of log of all problems")
for nhm,nl,elog in out:
    #print(elog.size())
    if elog.size()>0:
        print(elog.get_error_log())

    

List of log of all problems


We need to deal with a disconnect between how we created our source collection and the normal MsPASS expectation.  Specifically, we created this data set from css3.0 tables where source data ids are defined by two integer keys called "evid" and "orid".   We kept only "orid" for reasons that are not necessary for this class.   The problem is MsPASS wants to normally use the key "source_id" that is the ObjectId of a document in the source collection.   The following algorithm addresses that in a somewhat obscure way done for efficiency.   That is, it builds the update list and then its the database server all at once with "update_many".   Experience has shown that is orders of magnitude faster than single document transactions. 

In [21]:
# first build a cross-reference dictionary - rational because source is small
orid_xref=dict()
cursor = db.source.find({})
for doc in cursor:
    # type conversion is safer
    orid = int(doc['orid'])
    sid = doc['_id']
    orid_xref[orid] = sid
print("Size of orid cross-reference dictionary=",len(orid_xref))

Size of orid cross-reference dictionary= 342


In [22]:
from pymongo import UpdateOne
from bson.objectid import ObjectId
operations = list()
cursor = db.wf_TimeSeries.find({})
nfail = 0   # just count problems instead of echoing errors
for doc in cursor:
    wfid = doc['_id']
    if "orid" in doc:
        orid = int(doc["orid"])
        if orid in orid_xref:
            sid = orid_xref[orid]
            operations.append(
                UpdateOne(
                    {"_id": ObjectId(wfid)},      # The Filter
                    {"$set": {"source_id": sid}}     # The Update
                )
    )

# Send the entire batch to MongoDB at once
if operations:
    result = db.wf_TimeSeries.bulk_write(operations)

In [23]:
# TODO:  these are now in MsPASS - change when newer version is installed on geolab
from db_utils import MongoDBWorker,fetch_dbhandle
dbplugin = MongoDBWorker(mspass_client)
dask_client.register_plugin(dbplugin)

ModuleNotFoundError: No module named 'db_utils'

In [ ]:
from mspasspy.db.normalize import MiniseedMatcher,normalize
from mspasspy.algorithms.bundle import bundle_seed_data
from mspasspy.util.seismic import number_live,print_metadata

def run_bundle(query,dbname,channel_matcher=None,site_matcher=None):
    db = fetch_dbhandle(dbname)
    # the default is very slow as every ensemble will have to do this
    if channel_matcher is None:
        channel_matcher=MiniseedMatcher(db,attributes_to_load=['starttime', 'endtime', 'lat', 'lon', 'elev', '_id','hang','vang'])
    if site_matcher is None:
        site_matcher = MiniseedMatcher(db,
                                       collection='site',
                                       attributes_to_load=['starttime', 'endtime', 'lat', 'lon', 'elev', '_id'],
                                      )
        
    cursor = db.wf_TimeSeries.find(query)
    tsens = db.read_data(cursor,collection="wf_TimeSeries")
    #print("ts ensemble size=",len(tsens.member))
    tsens = normalize(tsens,channel_matcher,handles_ensembles=False)
    #print(tsens.live)
    # this fixes an old bug from the old version being run here
    tsens.set_live()
    #print("Number live after normalize=",number_live(tsens))
    #print_metadata(tsens.member[0])
    ens = bundle_seed_data(tsens)
    del tsens
    #print("bundle output size=",len(ens.member))
    #print("ensemble live=",ens.live)
    #print(ens.elog.get_error_log())
    ens = normalize(ens,site_matcher,handles_ensembles=False)
    # this old version had a bug in normalize that was fixed 
    # this is a patch until a new version can be installed on geolab
    ens.set_live()
    #print("After normalize live=",ens.live)
    #print("elog size=",ens.elog.size())
    #print(ens.elog.get_error_log())
    ntotal = len(ens.member)
    nlive = number_live(ens)
    #for d in ens.member:
    #    print(d.live,d.elog.size())
    #print_metadata(ens.member[0])
    ret=db.save_data(ens,
                     storage_mode="file",
                     collection="wf_Seismogram", 
                     dir="wf_Seismogram",
                     dfile="ES26raw_seismograms.dat",
                    )
    return ntotal,nlive

In [ ]:
channel_matcher=MiniseedMatcher(db,attributes_to_load=['starttime', 'endtime', 'lat', 'lon', 'elev', '_id','hang','vang'])
site_matcher = MiniseedMatcher(db,
                                collection='site',
                                attributes_to_load=['starttime', 'endtime', 'lat', 'lon', 'elev', '_id'],
                            )
        

In [ ]:
from mspasspy.algorithms.bundle import bundle_seed_data
# extract a list of source id values that actually have data
srcidlist = db.wf_TimeSeries.distinct("source_id")
# process by ensemble 
# a subtle feature of MsPASS is default storage of ensembles are faster to read than atomic processing
querylist=list()
for srcid in srcidlist:
    query={"source_id" : srcid}
    querylist.append(query)
print("Number of ensembles to be processed = ",len(querylist))

In [ ]:
t0=time.time()
out = sliding_window_pipeline(querylist,
                              run_bundle,
                              dask_client,
                              pfunc_args=[db.name,channel_matcher,site_matcher],
                              verbose=True,
                             progress_report_interval=1000,
                             )
t=time.time()
print("Time to process with run_bundle=",t-t0)

In [ ]:
print("N N_live")
for nt,nl in out:
    print(nt,nl)